# CODE 0.1: Check the Environment

> **Goal:** make sure this computer is ready for the project: the right Python, the right library versions, and an NVIDIA GPU that PyTorch can use, also for an LSTM (the deep learning model of notebook 4.2).
>
> - **Input:** `requirements.txt` and `requirements-torch.txt` (the library versions this project needs)
> - **Output:** no files. Only a list of checks. At the end you should see **ALL CHECKS PASSED**.
>
> Run this notebook first, and again after you install or update a library. How to install everything is explained in the README, section **Setup (Windows)**.

### Words Used in This Notebook

> - **Virtual environment (venv):** a private folder with its own Python and its own libraries, for one project. Other projects keep their own libraries, so nothing clashes. Ours is the folder `.venv` in the project folder.
> - **Kernel:** the Python that runs the notebook. In VS Code you pick it at the top right of the notebook. It must be the `.venv` one.
> - **GPU:** the graphics card. It has thousands of small cores that do many calculations at the same time, so training a model on it is much faster than on the CPU.
> - **CUDA:** NVIDIA's software that lets programs use the GPU. PyTorch comes in two builds: one that only uses the CPU, and one with CUDA. We need the CUDA build.
> - **VRAM:** the GPU's own memory. The data and the model must fit in it while the model trains.
> - **Driver:** the program that lets Windows talk to the GPU. The command `nvidia-smi` in a terminal shows the driver and the GPU.
> - **LSTM:** a deep learning model that reads a sequence one step at a time, like reading a sentence word by word, and keeps a short memory of what it has read so far. In this project the sequence is the sensor readings of the compressor over time.

## Import Libraries & Variables

### Import Libraries

> Only libraries that come with Python, so this notebook also runs when other libraries are still missing. PyTorch is imported later, in "Check the GPU".
>
> - `os`: works with folders and file paths.
> - `sys`, `platform`: tell which Python and which computer run this notebook.
> - `time`: measures how long something takes.
> - `importlib.metadata`: reads the version of each installed library.

In [1]:
import os
import sys
import time
import platform
from importlib.metadata import version, PackageNotFoundError

### Variables

#### Folder Paths

> This code finds the main project folder. It starts in the folder of this notebook and goes up until it finds the file `requirements.txt`, which sits in the main project folder. So the paths work wherever the notebook is.
>
> - `ROOT_DIR`: the main project folder.
> - `REQUIREMENT_FILES`: the two files that list the library versions.

In [2]:
ROOT_DIR = os.getcwd()
while not os.path.isfile(os.path.join(ROOT_DIR, "requirements.txt")) and ROOT_DIR != os.path.dirname(ROOT_DIR):
    ROOT_DIR = os.path.dirname(ROOT_DIR)

REQUIREMENT_FILES = [os.path.join(ROOT_DIR, "requirements.txt"), os.path.join(ROOT_DIR, "requirements-torch.txt")]
print("Project folder found:", os.path.basename(ROOT_DIR))

Project folder found: predictive-maintenance-metropt


#### Expected Values

> - `MIN_PYTHON`: the library versions in `requirements.txt` need Python 3.12 or newer. This project uses Python 3.14.
> - `TEST_SIZE`: the size of the two number tables in the speed test (4096 x 4096).
> - `LSTM_WINDOWS`, `LSTM_STEPS`, `LSTM_SENSORS`: the shape of the random test data for the LSTM check. It looks like our real data will: 32 pieces of 30 minutes, with a reading every 10 seconds (180 time steps), and 15 sensors.
> - `LSTM_MEMORY`: how many numbers the test LSTM uses as its memory.
> - `SEED`: makes the random numbers the same every time the notebook runs.

In [3]:
MIN_PYTHON = (3, 12)
TEST_SIZE = 4096
LSTM_WINDOWS = 32
LSTM_STEPS = 180
LSTM_SENSORS = 15
LSTM_MEMORY = 64
SEED = 0

## Functions

### Record a Check

> `check()` prints PASS or FAIL for one check and stores the result in `results`, with the name of the check as the key.

In [4]:
results = {}

def check(name, ok):
    results[name] = bool(ok)
    print(("PASS  " if ok else "FAIL  ") + name)

### Read the Needed Versions

> `read_requirements()` reads one requirements file and returns each library with the version it needs. Lines like `numpy==2.5.4` are read. Comments (after `#`) and lines without `==` are skipped.

In [5]:
def read_requirements(path):
    needed = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.split("#")[0].strip()
            if "==" in line:
                name, wanted = line.split("==")
                needed[name.strip()] = wanted.strip()
    return needed

### Installed Version

> `installed_version()` returns the installed version of a library, or `not installed`.

In [6]:
def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return "not installed"

## Check Python

### Which Python Runs This Notebook?

> - `Python program` is the file that runs this notebook. It should end with `.venv\Scripts\python.exe`, the Python of the project's virtual environment. If it is `C:\Python314\python.exe` (the Python of the whole computer), pick the `.venv` kernel at the top right of the notebook.
> - A virtual environment is active when Python's own folder (`sys.prefix`) is different from the folder of the Python it was made from (`sys.base_prefix`).

In [7]:
print("Python version:", platform.python_version())
print("Python program:", sys.executable)
print("Computer:", platform.system(), platform.release(), "|", platform.machine())
print()

check(f"Python is {MIN_PYTHON[0]}.{MIN_PYTHON[1]} or newer", sys.version_info[:2] >= MIN_PYTHON)
check("this notebook runs inside a virtual environment", sys.prefix != sys.base_prefix)

Python version: 3.14.2
Python program: c:\Users\Kelvin Jonathan\Downloads\predictive-maintenance-metropt\.venv\Scripts\python.exe
Computer: Windows 11 | AMD64

PASS  Python is 3.12 or newer
PASS  this notebook runs inside a virtual environment


## Check the Libraries

### Are the Right Versions Installed?

> For every library in the two requirements files, we compare the needed version with the installed one. PyTorch from the PyTorch website adds the CUDA version to its own version, like `2.15.0+cu132`, so only the part before `+` is compared.

In [8]:
for path in REQUIREMENT_FILES:
    print(f"--- {os.path.basename(path)} ---")
    for name, wanted in read_requirements(path).items():
        have = installed_version(name)
        check(f"{name}: needs {wanted}, installed {have}", have.split("+")[0] == wanted)

--- requirements.txt ---
PASS  numpy: needs 2.5.4, installed 2.5.4
PASS  pandas: needs 3.1.1, installed 3.1.1
PASS  pyarrow: needs 26.0.1, installed 26.0.1
PASS  scipy: needs 1.18.1, installed 1.18.1
PASS  scikit-learn: needs 1.9.1, installed 1.9.1
PASS  shap: needs 0.54.1, installed 0.54.1
PASS  matplotlib: needs 3.11.2, installed 3.11.2
PASS  ipykernel: needs 7.4.0, installed 7.4.0
PASS  tqdm: needs 4.70.1, installed 4.70.1
PASS  streamlit: needs 1.67.0, installed 1.67.0
--- requirements-torch.txt ---
PASS  torch: needs 2.15.0, installed 2.15.0+cu132


## Check the GPU

### Can PyTorch Use the GPU?

> - `CUDA in this PyTorch build`: `None` means the CPU-only build is installed. Then PyTorch can never use the GPU.
> - `GPU found by PyTorch`: `True` means PyTorch can talk to the GPU through the driver.

In [9]:
try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("CUDA in this PyTorch build:", torch.version.cuda)
    print("GPU found by PyTorch:", torch.cuda.is_available(), "\n")
    check("PyTorch is the CUDA build (not the CPU-only build)", torch.version.cuda is not None)
    check("PyTorch can use the GPU", torch.cuda.is_available())
except ImportError:
    torch = None
    print("PyTorch is not installed. See README > Setup (Windows).\n")
    check("PyTorch is installed", False)

PyTorch version: 2.15.0+cu132
CUDA in this PyTorch build: 13.2
GPU found by PyTorch: True 

PASS  PyTorch is the CUDA build (not the CPU-only build)
PASS  PyTorch can use the GPU


### Which GPU?

> The name, the VRAM and the *compute capability*: a version number for what the GPU can do. An RTX 50 series GPU (NVIDIA's Blackwell generation) has compute capability 12.0.

In [10]:
if torch is not None and torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print(f"VRAM: {gpu.total_memory / 1024**3:.1f} GB")
    print(f"Compute capability: {gpu.major}.{gpu.minor}\n")
    check("the GPU is an NVIDIA GPU", "NVIDIA" in gpu.name)
else:
    print("No GPU to show. Fix the checks above first.")

GPU: NVIDIA GeForce RTX 5070 Laptop GPU
VRAM: 8.0 GB
Compute capability: 12.0

PASS  the GPU is an NVIDIA GPU


### Speed Test: CPU against GPU

> Training a model is mostly multiplying big tables of numbers. So we multiply two tables of random numbers (`TEST_SIZE` x `TEST_SIZE`) on the CPU and on the GPU, and time both. The GPU runs the multiplication once before the timing, because the first run also starts the GPU up. The GPU also works in the background while Python goes on, so `torch.cuda.synchronize()` waits until the GPU has really finished: once before the timer starts, and once before it stops.
>
> At the end we check that the GPU gives the same answer as the CPU (up to tiny rounding differences).

In [11]:
if torch is not None and torch.cuda.is_available():
    a = torch.rand(TEST_SIZE, TEST_SIZE)
    b = torch.rand(TEST_SIZE, TEST_SIZE)

    start = time.perf_counter()
    cpu_answer = a @ b
    cpu_seconds = time.perf_counter() - start

    a_gpu, b_gpu = a.cuda(), b.cuda()
    a_gpu @ b_gpu
    torch.cuda.synchronize()
    start = time.perf_counter()
    gpu_answer = a_gpu @ b_gpu
    torch.cuda.synchronize()
    gpu_seconds = time.perf_counter() - start

    print(f"CPU: {cpu_seconds * 1000:.0f} ms | GPU: {gpu_seconds * 1000:.1f} ms | the GPU is {cpu_seconds / gpu_seconds:.0f} times faster\n")
    check("the GPU gives the same answer as the CPU", torch.allclose(gpu_answer.cpu(), cpu_answer, rtol=1e-3, atol=1e-2))
else:
    print("Speed test skipped: PyTorch cannot use the GPU yet.")

CPU: 360 ms | GPU: 15.7 ms | the GPU is 23 times faster

PASS  the GPU gives the same answer as the CPU


### Can the GPU Run an LSTM?

> Notebook 4.2 trains an LSTM on the GPU. On NVIDIA GPUs, PyTorch runs an LSTM with **cuDNN**, NVIDIA's library of fast building blocks for deep learning. So we test one small LSTM now, before we need it:
>
> 1. Make random test data shaped like our real data: `LSTM_WINDOWS` pieces, each `LSTM_STEPS` time steps long, with `LSTM_SENSORS` sensors.
> 2. Make one small LSTM. Its weights are random, because we only test that it runs, not that it is good.
> 3. Let the CPU and the GPU read the same data with the same LSTM, and compare their answers.
>
> For every time step of every piece, the LSTM gives `LSTM_MEMORY` numbers: its memory of what it has read so far. These numbers are always between -1 and 1. The GPU may round a little differently than the CPU, so we allow a difference of up to 0.01.

In [12]:
if torch is not None and torch.cuda.is_available():
    torch.manual_seed(SEED)
    lstm = torch.nn.LSTM(input_size=LSTM_SENSORS, hidden_size=LSTM_MEMORY, batch_first=True)
    windows = torch.rand(LSTM_WINDOWS, LSTM_STEPS, LSTM_SENSORS)

    with torch.no_grad():
        cpu_memory = lstm(windows)[0]
        gpu_memory = lstm.cuda()(windows.cuda())[0].cpu()
    largest_difference = (gpu_memory - cpu_memory).abs().max().item()

    print(f"Test data: {LSTM_WINDOWS} pieces x {LSTM_STEPS} time steps x {LSTM_SENSORS} sensors")
    print(f"LSTM answer: {tuple(gpu_memory.shape)} (pieces, time steps, memory numbers)")
    print(f"Largest difference between GPU and CPU: {largest_difference:.6f}\n")
    check("the GPU runs an LSTM with the same answer as the CPU", largest_difference < 0.01)
else:
    print("LSTM test skipped: PyTorch cannot use the GPU yet.")

Test data: 32 pieces x 180 time steps x 15 sensors
LSTM answer: (32, 180, 64) (pieces, time steps, memory numbers)
Largest difference between GPU and CPU: 0.000077

PASS  the GPU runs an LSTM with the same answer as the CPU


## Result

### Final Result

> If every check passed, the computer is ready. If a check failed, its name is printed.

In [13]:
failed = [name for name, ok in results.items() if not ok]
if results and not failed:
    print(f"ALL CHECKS PASSED ({len(results)} checks)")
else:
    print(f"{len(failed)} of {len(results)} checks FAILED:")
    for name in failed:
        print("  -", name)

ALL CHECKS PASSED (18 checks)


### What to Do When a Check Fails

> - **Python is not 3.12 or newer, or not inside a virtual environment:** pick the `.venv` kernel at the top right of the notebook. If there is no `.venv`, make it first (README > Setup (Windows), steps 2 to 5).
> - **A library has the wrong version or is not installed:** in the terminal of VS Code, run `.venv\Scripts\python.exe -m pip install -r requirements.txt`.
> - **PyTorch is not installed, or it is the CPU-only build:** remove it and install the CUDA build:
>   `.venv\Scripts\python.exe -m pip uninstall -y torch`, then
>   `.venv\Scripts\python.exe -m pip install -r requirements-torch.txt`
> - **PyTorch cannot use the GPU, but it is the CUDA build:** update the NVIDIA driver (NVIDIA App > Drivers), restart the laptop, and run this notebook again. In a terminal, `nvidia-smi` must show your GPU.
>
> After a fix: restart the kernel (Restart button at the top of the notebook) and run all cells again.

## Summary

> - This notebook checks the Python version, the virtual environment, every library version, and whether PyTorch can use the NVIDIA GPU.
> - It ends with two GPU tests: a speed test, so you can see how much faster the GPU is than the CPU, and a small LSTM, the type of model that notebook 4.2 trains.
>
> **Next:** `notebooks/1_data_setup/1.1_download_and_check_data.ipynb`